# 4th-power alias limit and the false-clean metric trap

This notebook is the slower companion to `reports/qpsk-alias-limit.md`.

The narrow question is the useful one: once the 4th-power coarse frequency estimate crosses its `\pi/4` alias limit, what actually fails first?


## The identity behind the cliff

For QPSK, the coarse frequency estimator works on `s[n]^4`, so it sees `4 \omega` instead of `\omega`.

That makes the estimate unique only while

```text
|4 ω| < π
```

which is the same as

```text
|ω| < π / 4
```

Beyond that boundary, the estimator folds onto the wrong branch and leaves a wrapped residual close to `±π/2` rad/sample after coarse correction.


In [ ]:
import csv
from pathlib import Path

rows = []
with (Path('..') / 'assets' / 'qpsk-alias-limit-map.csv').open() as handle:
    for row in csv.DictReader(handle):
        rows.append({key: float(value) for key, value in row.items()})
len(rows), rows[0].keys()


## Where the estimate folds

Pick a few offsets around the boundary. The clean case should have a near-zero wrapped residual. The aliased case should jump to about `π/2`.


In [ ]:
for target in [0.70, 0.75, 0.80, 0.85]:
    row = min(rows, key=lambda row: abs(row['freq_offset'] - target))
    print(target, {
        'estimate': round(row['coarse_frequency_estimate'], 3),
        'wrapped_residual': round(row['wrapped_residual_frequency'], 3),
        'tracked_rms': round(row['freq_acquired_tracked_rms_error'], 3),
        'symbol_ser': round(row['freq_acquired_symbol_error_rate'], 3),
    })


That is the whole story in miniature.

- below `π/4`, the wrapped residual stays near zero and the best static-quadrant SER stays near zero
- above `π/4`, the residual jumps to roughly `π/2` and the SER jumps with it

But the old RMS metric does **not** jump.


In [ ]:
[(round(row['freq_offset'], 2), round(row['freq_acquired_tracked_rms_error'], 3), round(row['freq_acquired_symbol_error_rate'], 3))
 for row in rows if abs(abs(row['freq_offset']) - 0.8) < 0.06]


## Why nearest-corner RMS lies here

The metric only asks whether the tracked samples land near *some* QPSK corner.

A residual near `π/2` rad/sample can still hop corner to corner cleanly, so the cloud looks tidy. What fails is the fixed symbol labeling over time.

That is why this notebook uses **best static-quadrant symbol error rate** instead: it forgives one constant 90-degree ambiguity, but it does not forgive a quarter-turn ramp from symbol to symbol.


## Takeaway

The alias boundary is not just an estimator footnote. It creates a real evaluation trap.

1. the 4th-power estimate stays honest up to `|ω| < π/4`
2. beyond that, the coarse estimate folds by a quarter-turn-per-symbol branch error
3. nearest-corner RMS can still look clean after tracking, so you need a symbol-agreement metric that factors out only a *static* quadrant ambiguity

That is the reason this sidecar belongs in the repo instead of living as a one-line caveat.
